In [1]:
from typing import TypedDict
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph,START,END

In [ ]:

llm = ChatGroq(model="openai/gpt-oss-120b",api_key="X")

In [8]:
class EmailState(TypedDict):
    customer_email: str
    extracted_issue: str
    english_reply: str
    hinglish_reply: str

In [9]:
def extract_issue(state: EmailState):
    prompt = f"""
    You are a customer support analyst.

    Read the customer email and identify the main problem.

    Rules:
    - Summarize the core issue in one short sentence.
    - Focus on the actual problem, not unnecessary details.
    - Do not assume facts that are not mentioned.
    - Do not include greetings or explanations.

    Customer Email:
    {state['customer_email']}

    Output only the issue summary.
    """
    return {"extracted_issue": llm.invoke(prompt).content.strip()}


def draft_reply(state: EmailState):
    prompt = f"""
    You are a professional and empathetic customer support agent.

    Write a reply addressing the customer's issue.

    Rules:
    - Use polite, simple, and professional English.
    - Start with a brief apology or acknowledgment when appropriate.
    - Provide a relevant solution or clear next step.
    - Never promise actions or results that are not confirmed.
    - Keep the reply within 2-3 short sentences.
    - Do not add a subject line or unnecessary greetings.

    Customer Issue:
    {state['extracted_issue']}

    Output only the customer support reply.
    """
    return {"english_reply": llm.invoke(prompt).content.strip()}


def translate_to_hinglish(state: EmailState):
    prompt = f"""
    You are an expert in natural Hindi-English communication.

    Convert the following customer support reply into polite,
    professional Hinglish.

    Rules:
    - Use natural Roman Hindi mixed with simple English words.
    - Preserve the original meaning and solution exactly.
    - Keep the tone respectful, empathetic, and friendly.
    - Do not translate technical terms unnecessarily.
    - Do not add new promises or information.
    - Keep the same number of sentences as the original reply.

    English Reply:
    {state['english_reply']}

    Output only the Hinglish reply in Roman script.
    """
    return {"hinglish_reply": llm.invoke(prompt).content.strip()}

In [10]:
builder=StateGraph(EmailState)

In [11]:
builder.add_node("extracter",extract_issue)
builder.add_node("drafter",draft_reply)
builder.add_node("translatter",translate_to_hinglish)

builder.add_edge(START,"extracter")
builder.add_edge("extracter","drafter")
builder.add_edge("drafter","translatter")
builder.add_edge("translatter",END)



In [12]:
graph=builder.compile()

In [15]:
email="""Hello Support Team,

I placed an order yesterday and the payment was successfully deducted from my bank account. However, I have not received any order confirmation email, and the order is not showing in my account.

I am worried that my payment has been lost. Please check the payment status and either confirm my order or help me get a refund.

My transaction ID is TXN123456789.

Thank you for your help.

Regards,
Rahul"""
result=graph.invoke({"customer_email":email})

In [16]:
print(result["extracted_issue"])
print(result["english_reply"])
print(result["hinglish_reply"])

Payment was deducted but no order confirmation or record appears, requiring verification and possible refund.
We’re sorry to hear your payment was deducted without an order confirmation. Please share the transaction ID or receipt so we can verify the payment and process a refund if needed. We’ll investigate and update you shortly.
Humein bahut afsos hai ki aapka payment order confirmation ke bina deduct ho gaya. Kripya humein transaction ID ya receipt share karen, taaki hum payment verify karke zaroorat padne par refund process kar saken. Hum iski janch karenge aur jald hi aapko update karenge.
